In [19]:
%pip install --upgrade pip langchain datasets langchain-anthropic langchain-huggingface "langchain-community<=0.4" pandas langchain-qdrant langchain-ollama langchain-core langchain-openai langchain-text-splitters python-dotenv sentence-transformers anthropic ragas "qdrant-client[fastembed]>=1.14.1"



  Using cached langchain_anthropic-1.4.8-py3-none-any.whl.metadata (3.5 kB)
Using cached langchain_anthropic-1.4.8-py3-none-any.whl (52 kB)
Note: you may need to restart the kernel to use updated packages.


In [1]:
import os
from dotenv import load_dotenv
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import TextLoader
from qdrant_client.models import Distance, VectorParams
from langchain_qdrant import QdrantVectorStore
from qdrant_client import QdrantClient
from langchain_openai import OpenAIEmbeddings
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_ollama import ChatOllama
from ragas import EvaluationDataset

from ragas import evaluate
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper

from fastembed import TextEmbedding
from fastembed.rerank.cross_encoder import TextCrossEncoder

load_dotenv()

/home/jonas/dev/holmes-comparison-grag-rag/local-rag/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

In [2]:
TextCrossEncoder.list_supported_models()

[{'model': 'Xenova/ms-marco-MiniLM-L-6-v2',
  'sources': {'hf': 'Xenova/ms-marco-MiniLM-L-6-v2',
   'url': None,
   '_deprecated_tar_struct': False},
  'model_file': 'onnx/model.onnx',
  'description': 'MiniLM-L-6-v2 model optimized for re-ranking tasks.',
  'license': 'apache-2.0',
  'size_in_GB': 0.08,
  'additional_files': []},
 {'model': 'Xenova/ms-marco-MiniLM-L-12-v2',
  'sources': {'hf': 'Xenova/ms-marco-MiniLM-L-12-v2',
   'url': None,
   '_deprecated_tar_struct': False},
  'model_file': 'onnx/model.onnx',
  'description': 'MiniLM-L-12-v2 model optimized for re-ranking tasks.',
  'license': 'apache-2.0',
  'size_in_GB': 0.12,
  'additional_files': []},
 {'model': 'BAAI/bge-reranker-base',
  'sources': {'hf': 'BAAI/bge-reranker-base',
   'url': None,
   '_deprecated_tar_struct': False},
  'model_file': 'onnx/model.onnx',
  'description': 'BGE reranker base model for cross-encoder re-ranking.',
  'license': 'mit',
  'size_in_GB': 1.04,
  'additional_files': []},
 {'model': 'jinaa

In [78]:
from langchain_huggingface import HuggingFaceEmbeddings

#

model_name = "BAAI/bge-large-en"
model_kwargs = {"device": "cpu"}
encode_kwargs = {"normalize_embeddings": False}
embeddings = HuggingFaceEmbeddings(
    model_name=model_name, model_kwargs=model_kwargs, encode_kwargs=encode_kwargs
)

# embeddings = HuggingFaceEmbeddings(
#     model_name="BAAI/bge-large-en-v1.5",
#     model_kwargs={"device": "cpu"},
#     encode_kwargs={"normalize_embeddings": True},
#     query_encode_kwargs={
#         "prompt": "Represent this sentence for searching relevant passages: ",
#         "normalize_embeddings": True,
#     },
# )

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 2971.77it/s]


In [64]:
documents = []
directory = "../data/chapters/"
for file in os.listdir(directory):
    loader = TextLoader(file_path=f"../data/chapters/{file}")
    docs = loader.load()

    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=1200,
        chunk_overlap=150,
        add_start_index=True,
    )
    all_splits = text_splitter.split_documents(docs)
    documents.extend(all_splits)

    print(f"Split {file} into {len(all_splits)} sub-documents.")

Split chapter_8.txt into 58 sub-documents.
Split chapter_4.txt into 59 sub-documents.
Split chapter_1.txt into 53 sub-documents.
Split chapter_6.txt into 58 sub-documents.
Split chapter_5.txt into 46 sub-documents.
Split chapter_7.txt into 43 sub-documents.
Split chapter_12.txt into 61 sub-documents.
Split chapter_3.txt into 43 sub-documents.
Split chapter_2.txt into 52 sub-documents.
Split chapter_9.txt into 49 sub-documents.
Split chapter_11.txt into 56 sub-documents.
Split chapter_10.txt into 50 sub-documents.


In [28]:
len(documents)

481

## Hybrid retrieval with reranking vector store

In [65]:
from qdrant_client import QdrantClient, models
from qdrant_client.models import Distance, VectorParams, Document, PointStruct

url = "http://localhost:6333"
client = QdrantClient(url=url)

In [66]:
dense_embedding_model = "sentence-transformers/all-MiniLM-L6-v2"
sparse_embedding_model = "qdrant/bm25"
late_interaction_embedding_model = "answerdotai/answerai-colbert-small-v1"

In [67]:
collection_name = "local-hybrid-search"

if client.collection_exists(collection_name=collection_name):
    client.delete_collection(collection_name=collection_name)

client.create_collection(
    collection_name,
    vectors_config={
        "dense": models.VectorParams(
            size=384,
            distance=models.Distance.COSINE,
        ),
        "multi": models.VectorParams(
            size=96,
            distance=models.Distance.COSINE,
            multivector_config=models.MultiVectorConfig(
                comparator=models.MultiVectorComparator.MAX_SIM,
            ),
            hnsw_config=models.HnswConfigDiff(m=0)  #  Disable HNSW for reranking
        ),
    },
    sparse_vectors_config={
        "sparse": models.SparseVectorParams(modifier=models.Modifier.IDF)
    }
)

True

In [68]:
points = (  
    PointStruct(
        id=idx,
        vector={
            "dense": Document(text=doc.page_content, model=dense_embedding_model),
            "sparse": Document(text=doc.page_content, model=sparse_embedding_model),
            "multi": Document(
                text=doc.page_content, model=late_interaction_embedding_model
            ),
        },
        payload={"page_content": doc.page_content, "metadata": doc.metadata},
    )
    for idx, doc in enumerate(documents)
)
client.upload_points(collection_name=collection_name, points=points, batch_size=25)

In [69]:
def retrieve_with_reranking(query, k=10):
    prefetch = [
        models.Prefetch(
            query=models.Document(text=query, model=dense_embedding_model),
            using="dense",
            limit=300,
        ),
        models.Prefetch(
            query=models.Document(text=query, model=sparse_embedding_model),
            using="sparse",
            limit=300,
        ),
    ]

    results = client.query_points(
        collection_name,
        prefetch=prefetch,
        query=models.Document(text=query, model=late_interaction_embedding_model),
        using="multi",
        with_payload=True,
        limit=k,
    )
    return results.points

In [72]:
llm = ChatOllama(
    model="qwen3.6:latest",
    base_url="http://192.168.178.67:11434",
    temperature=0,
    reasoning=False,
    keep_alive="24h",
    # timeout=600,
    # verbose=False,
    # num_ctx=8192,
    # disable_streaming=False,
)

llm.invoke("Say hello in up to three words!")

AIMessage(content='Hello there!', additional_kwargs={}, response_metadata={'model': 'qwen3.6:latest', 'created_at': '2026-07-09T10:08:07.2374836Z', 'done': True, 'done_reason': 'stop', 'total_duration': 1352316300, 'load_duration': 531615100, 'prompt_eval_count': 20, 'prompt_eval_duration': 321945000, 'eval_count': 4, 'eval_duration': 493594000, 'logprobs': None, 'model_name': 'qwen3.6:latest', 'model_provider': 'ollama'}, id='lc_run--019f4659-52a4-77a3-901a-ca8566292738-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 20, 'output_tokens': 4, 'total_tokens': 24})

In [73]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# retriever = vector_store.as_retriever(search_type="similarity", search_kwargs={"k": 5, "score_threshold": 0.3})

prompt_template = """
You are an assistant that answers questions using only the provided context.

Rules:
- Answer with the shortest possible correct answer.
- Output only the answer itself.
- Do not explain your answer.
- Do not repeat the question.
- Do not add background, reasoning, or extra words.
- Do not quote the context unless the answer is a direct quote.
- Use only information explicitly stated in the context.
- If the answer is not fully supported by the context, respond exactly with: I cannot answer based on the context.

Context:
{context}

Question:
{query}

Answer:
"""
prompt = ChatPromptTemplate.from_template(prompt_template)

In [74]:
def format_docs(relevant_docs):
    return "\n".join(doc.payload["page_content"] for doc in relevant_docs)
    # return "\n".join(doc.page_content for doc in relevant_docs)

def answer_with_context(query):
    # docs = retriever.invoke(query)
    docs =  retrieve_with_reranking(query, k=10)
    context = format_docs(docs)

    answer = (prompt | llm | StrOutputParser()).invoke({
        "context": context,
        "query": query
    })

    return {
        "query": query,
        "context": context,
        "answer": answer,
        "docs": docs,
    }

# query = "Which House does the King of Bohemia belong to?"
# answer_with_context(query)

# relevant_docs = retriever.invoke(query)
# print(relevant_docs)
# chain.invoke({"context": format_docs(relevant_docs), "query": query})
# chain.invoke(query)

In [75]:
import json

qa_path = "../data/qa/"
# for file in os.listdir(qa_path):
with open(f"{qa_path}holmes_qa_subset.json") as f:
    json_qa = json.loads(f.read())

len(json_qa)

8

In [76]:
qa_data = []
for pair in json_qa:
    query = pair["question"]
    response = answer_with_context(query)
    qa_data.append(
        {
            "user_input": pair["question"],
            "retrieved_contexts": [
                res.payload["page_content"] for res in response["docs"]
            ],
            # "retrieved_contexts": [res.page_content for res in response["docs"]],
            "response": response["answer"],
            "reference": pair["answer"],
        }
    )
    print(f"processed {len(qa_data)}/{len(json_qa[:10])} qa pairs")
evaluation_dataset = EvaluationDataset.from_list(qa_data)

processed 1/8 qa pairs
processed 2/8 qa pairs
processed 3/8 qa pairs
processed 4/8 qa pairs
processed 5/8 qa pairs
processed 6/8 qa pairs
processed 7/8 qa pairs
processed 8/8 qa pairs


In [18]:
qa_data

[{'user_input': "What is the name of the woman Holmes remembers most in 'A Scandal in Bohemia'?",
  'retrieved_contexts': ['I. A SCANDAL IN BOHEMIA\n\n\nI.',
   '“Very truly yours,\n\n    “IRENE NORTON, _née_ ADLER.”\n\n\n“What a woman—oh, what a woman!” cried the King of Bohemia, when we had\nall three read this epistle. “Did I not tell you how quick and resolute\nshe was? Would she not have made an admirable queen? Is it not a pity\nthat she was not on my level?”\n\n“From what I have seen of the lady, she seems, indeed, to be on a very\ndifferent level to your Majesty,” said Holmes coldly. “I am sorry that\nI have not been able to bring your Majesty’s business to a more\nsuccessful conclusion.”\n\n“On the contrary, my dear sir,” cried the King; “nothing could be more\nsuccessful. I know that her word is inviolate. The photograph is now as\nsafe as if it were in the fire.”\n\n“I am glad to hear your Majesty say so.”\n\n“I am immensely indebted to you. Pray tell me in what way I can re

In [ ]:
import pandas as pd

samples = [
    {
        "query": "What is the name of the woman Holmes remembers most in 'A Scandal in Bohemia'?",
        "grading_notes": "Irene Adler.",
    },
    {"query": "What animal contains the blue carbuncle?", "grading_notes": "A goose."},
    {
        "query": "Who is arrested in the Boscombe Valley mystery?",
        "grading_notes": "James McCarthy.",
    },
]
pd.DataFrame(samples).to_csv("../data/test_dataset.csv", index=False)

In [ ]:
# from openai import OpenAI
# from ragas.llms import llm_factory

# # Create an OpenAI-compatible client for Ollama
# client = OpenAI(
#     api_key="ollama",  # Ollama doesn't require a real key
#     base_url="http://192.168.178.67:11434/v1",
# )
# llm = llm_factory("qwen3.6:latest", provider="openai", client=client)

In [36]:
# from ragas import evaluate, RunConfig
# from ragas.metrics import Faithfulness, LLMContextRecall, ContextPrecision
# from ragas.llms import llm_factory
# from ragas.embeddings import embedding_factory
# from anthropic import Anthropic

# # Setup your evaluator LLM
# anthropic_client = Anthropic(api_key=os.getenv("ANTHROPIC_API_KEY"))
# evaluator_llm = llm_factory(
#     "claude-haiku-4-5-20251001",
#     provider="anthropic",
#     client=anthropic_client,
#     # max_tokens=2048,
#     temperature=0.0,
#     # timeout=600,
# )
# evaluator_embedding = embedding_factory("huggingface", "BAAI/bge-m3")

# # scorer = ContextPrecision(llm=evaluator_llm)
# metrics = [
#     # AnswerCorrectness(llm=evaluator_llm),
#     # AnswerRelevancy(llm=evaluator_llm),
#     # Faithfulness(llm=evaluator_llm),
#     # ContextRecall(llm=evaluator_llm),
#     # ContextPrecision(llm=evaluator_llm),
#     ContextPrecision(llm=evaluator_llm),
#     LLMContextRecall(llm=evaluator_llm),
#     Faithfulness(llm=evaluator_llm),
# ]

# config = RunConfig(timeout=600, max_retries=20, max_wait=50, log_tenacity=False)

# result = evaluate(
#     dataset=evaluation_dataset,
#     metrics=metrics,
#     llm=evaluator_llm,
#     embeddings=evaluator_embedding,
#     raise_exceptions=False,
#     show_progress=True,
#     batch_size=8,
#     run_config=config,
# )

# result = await scorer.ascore(
# #     user_input=qa_data[0]["user_input"],
# #     reference=qa_data[0]["reference"],
# #     retrieved_contexts=qa_data[0]["retrieved_contexts"],
# # )

In [ ]:
result

{'context_precision': nan, 'context_recall': nan, 'faithfulness': nan}

In [77]:
import pandas
from ragas import evaluate, RunConfig
from ragas.metrics import (
    answer_correctness,
    answer_relevancy,
    faithfulness,
    context_precision,
    context_recall,
)
from langchain_anthropic import ChatAnthropic

config = RunConfig(timeout=600, max_retries=20, max_wait=50, log_tenacity=False)
evaluation_llm = ChatAnthropic(
    model_name="claude-haiku-4-5-20251001",
    temperature=0,
    api_key=os.getenv("ANTHROPIC_API_KEY"),
)
judge_llm = LangchainLLMWrapper(evaluation_llm)
judge_embedding = LangchainEmbeddingsWrapper(embeddings)
scores = evaluate(
    evaluation_dataset,
    metrics=[
        answer_correctness,
        answer_relevancy,
        faithfulness,
        context_precision,
        context_recall,
    ],
    llm=judge_llm,
    embeddings=judge_embedding,
    run_config=config,
)

# print(rows)
print(scores)

/tmp/ipykernel_213399/921869118.py:3: DeprecationWarning: Importing answer_correctness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import answer_correctness
  from ragas.metrics import (
/tmp/ipykernel_213399/921869118.py:3: DeprecationWarning: Importing answer_relevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import answer_relevancy
  from ragas.metrics import (
/tmp/ipykernel_213399/921869118.py:3: DeprecationWarning: Importing faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import faithfulness
  from ragas.metrics import (
/tmp/ipykernel_213399/921869118.py:3: DeprecationWarning: Importing context_precision from 'ragas.metrics' is deprecated and will b

{'answer_correctness': 0.6793, 'answer_relevancy': 0.4426, 'faithfulness': 0.7500, 'context_precision': 0.3437, 'context_recall': 0.6250}


In [ ]:
scores

{'answer_correctness': nan, 'answer_relevancy': 0.5088, 'faithfulness': 0.0000, 'context_precision': nan, 'context_recall': 0.6667}